In [0]:
%sql
drop table if exists real_estate.silver.propiedades;

create table real_estate.silver.propiedades (
    propiedad_id bigint generated always as identity (start with 1 increment by 1) comment 'ID único de la propiedd (auto-generado)',

    partido string comment 'Partido/municipio estandarizado',
    region string comment 'Región geográfica: capital federal, gba zona norte/oeste/sur',

    tipo_de_operacion string comment 'alquiler, venta, alquiler_temporario',
    precio decimal(15,2) comment 'Precio de la propiedad',
    moneda string comment 'USD O ARS',
    expensas decimal(15,2) comment 'Expensas mensuales',

    ambientes int comment 'Cantidad de ambientes',
    metros_cuadrados_totales decimal(15,2) comment 'Superficie total',
    metros_cuadrados_cubiertos decimal(15,2) comment 'Superficie cubierta',

    antiguedad int comment 'Años de antiguedad (999 si no disponible)',
    cochera boolean comment 'Tiene cochera',
    orientacion string comment 'Orientación del inmueble',
    estado string comment 'Estado de la propiedad',
    url string comment 'URL de la propiedad',

    fecha_publicacion date comment 'Fecha de publicación — desde fecha (STRING) en Bronze',

    precio_por_m2 decimal(15,2) comment 'precio por metro cuadrado',

    _source_table string default 'real_estate.bronze.properties_bronze' comment 'Tabla origen',
    _processing_timestamp timestamp default current_timestamp comment 'Timestamp de procesamiento'
 )
 using delta 
 TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
 COMMENT 'Propiedades inmobiliarias - Capa Silver (datos limpios y validados)';

In [0]:
%sql
describe table extended real_estate.silver.propiedades;

In [0]:
%sql
-- propiedades_clean: casting básico de tipos 
create or replace temporary view propiedades_clean as 
select 
    case when precio rlike '^[^a-zA-Z]+$' then precio::double else null end as precio,
    moneda,
    case when ambientes rlike '^[^a-zA-Z]+$' then ambientes::double else null end as ambientes,
    case when metros_cuadrados_totales rlike '^[^a-zA-Z]+$' then metros_cuadrados_totales::double else null end as metros_cuadrados_totales,
    case when metros_cuadrados_cubiertos rlike '^[^a-zA-Z]+$' then metros_cuadrados_cubiertos::double else null end as metros_cuadrados_cubiertos,
    case when antiguedad rlike '^[^a-zA-Z]+$' then antiguedad::double else null end as antiguedad,
    tipo_de_operacion, id, ubicacion, numero, calle, expensas,
    orientacion_cardinal, orientacion_inmueble, piso, cochera, estado,
    tipo_vendedor, url, zona, fecha, hora
from real_estate.bronze.properties_bronze;

-- propiedades_clean_2: filtro de outliers por percentiles (Semana 2)
create or replace temporary view propiedades_clean_2 as 
(
  with limites as (
    select moneda, tipo_de_operacion,
      percentile(precio, 0.01) as p01,
      percentile(precio, 0.99) as p99
    from propiedades_clean
    where precio > 0 
      and moneda in ('ARS', 'USD')
      and tipo_de_operacion in ('venta', 'alquiler')
    group by moneda, tipo_de_operacion
  )
  select p.*
  from propiedades_clean p 
  join limites l
  on p.moneda = l.moneda
  and p.tipo_de_operacion = l.tipo_de_operacion
  where p.precio between l.p01 and l.p99
);

-- bronze_EDA: limpieza detallada 
create or replace temporary view bronze_EDA as (
    select 
        edl.id,
        edl.ubicacion,
        case 
          when edl.precio = 'NaN' or edl.precio not rlike '^-?[0-9]+\\.?[0-9]*$' then null
          when edl.precio::float between 0 and 2147483648 then edl.precio::float 
          else null
        end as precio,
        case 
          when edl.numero = 'NaN' or edl.numero not rlike '^-?[0-9]+\\.?[0-9]*$' then null
          when edl.numero::float between -10000 and 50000 then edl.numero::float
          else null
        end as numero,
        case 
          when edl.expensas = 'NaN' or edl.expensas not rlike '^-?[0-9]+\\.?[0-9]*$' then null 
          when edl.expensas::float  between 0 and 2000000 then edl.expensas::float 
          else null 
        end as expensas,
        edl.tipo_de_operacion,
        case 
          when lower(edl.moneda) like '%dolares%' then 'USD'
          when lower(edl.moneda) like '%us%' then 'USD'
          when lower(edl.moneda) like '%pesos%' then 'ARS'
          when lower(edl.moneda) like '%ars' then 'ARS'
          else edl.moneda
        end as moneda,
        case 
          when edl.ambientes = 'NaN' or edl.ambientes not rlike '^-?[0-9]+\\.?[0-9]*$' then null 
          else edl.ambientes::float
        end as ambientes,
        case 
          when edl.metros_cuadrados_totales = 'NaN' or edl.metros_cuadrados_totales not rlike '^-?[0-9]+\\.?[0-9]*$' then null
          else edl.metros_cuadrados_totales::float 
        end as metros_cuadrados_totales,
        case
          when edl.metros_cuadrados_cubiertos = 'NaN' or edl.metros_cuadrados_cubiertos not rlike '^-?[0-9]+\\.?[0-9]*$' then null
          else edl.metros_cuadrados_cubiertos::float
        end as metros_cuadrados_cubiertos,
        edl.orientacion_cardinal,
        edl.orientacion_inmueble,
        case 
          when edl.piso is null then null
          when edl.piso = 'NaN' then null
          else edl.piso::float
        end as piso,
        case 
          when edl.cochera = 'tiene' then 1 
          else null 
        end as cochera,
        edl.antiguedad,
        edl.estado,
        edl.tipo_vendedor,
        edl.url,
        edl.zona,
        coalesce(cast(edl.fecha as date), current_date()) as fecha
    from propiedades_clean_2 edl
    where edl.url rlike 'https'
      and (edl.zona like '%gba%' or edl.zona like '%caba%' or edl.zona like '%capital%')
      and edl.zona like '%-%'
      and length(edl.zona) < 50
      
);

select count(*) as registros_bronze_eda from bronze_EDA
